In [1]:
from datasets import load_dataset
import pandas as pd

dataset = load_dataset("squad")
df = pd.DataFrame(dataset["validation"])

df_small = df.sample(n=300, random_state=42).reset_index(drop=True)
df_small = df_small[["id", "title", "context", "question", "answers"]]

df_small.to_csv("squad_val_300.csv", index=False)

print(len(df_small))
print(df_small.head())

300
                         id                                      title  \
0  571cd3b55efbb31900334e04                                     Oxygen   
1  57097d63ed30961900e841fd                       Sky_(United_Kingdom)   
2  572754cd5951b619008f8866                               Genghis_Khan   
3  572855973acd2414000df929                      University_of_Chicago   
4  57293e221d046914007791d5  Intergovernmental_Panel_on_Climate_Change   

                                             context  \
0  Hyperbaric (high-pressure) medicine uses speci...   
1  The agreements include fixed annual carriage f...   
2  The invasions of Baghdad, Samarkand, Urgench, ...   
3  The university runs a number of academic insti...   
4  The executive summary of the WG I Summary for ...   

                                            question  \
0  What does increased oxygen concentrations in t...   
1  Does the new deal include Video on demand and ...   
2  In which year did Genghis Khan's grandson i

In [2]:
import ast
import pandas as pd

df = pd.read_csv("squad_val_300.csv")

def get_first_answer(ans):
    if isinstance(ans, str):
        ans = ast.literal_eval(ans)
    return ans["text"][0] if len(ans["text"]) > 0 else ""

df["gold_answer"] = df["answers"].apply(get_first_answer)

df.to_csv("squad_val_300.csv", index=False)
print(df[["question", "gold_answer"]].head())

                                            question  \
0  What does increased oxygen concentrations in t...   
1  Does the new deal include Video on demand and ...   
2  In which year did Genghis Khan's grandson inva...   
3         Where is the Hyde Park Day School located?   
4  What does the WG I Summary for Policymakers re...   

                                         gold_answer  
0                                    carbon monoxide  
1                                                 no  
2                                               1237  
3                   the University of Chicago campus  
4  substantially increasing the atmospheric conce...  


In [7]:
from openai import OpenAI

client = OpenAI(api_key=" ")

def squad_en_basic(context, question):
    prompt = f"""Answer the question using only a short span copied from the context.

Rules:
- Return only the shortest possible answer phrase.
- Do not write a full sentence.
- Do not explain.
- If the answer is yes or no, return only yes or no.

Context:
{context}

Question:
{question}
"""
    response = client.chat.completions.create(
        model="gpt-4o",
        temperature=0,
        messages=[{"role": "user", "content": prompt}]
    )

    return response.choices[0].message.content.strip()

In [8]:
for i in range(5):
    pred = squad_en_basic(df.loc[i, "context"], df.loc[i, "question"])
    print("Q:", df.loc[i, "question"])
    print("Pred:", pred)
    print("Gold:", df.loc[i, "gold_answer"])
    print("-----")

Q: What does increased oxygen concentrations in the patient's lungs displace?
Pred: carbon monoxide
Gold: carbon monoxide
-----
Q: Does the new deal include Video on demand and High Definition?
Pred: no
Gold: no
-----
Q: In which year did Genghis Khan's grandson invade Kievan Rus'?
Pred: 1237
Gold: 1237
-----
Q: Where is the Hyde Park Day School located?
Pred: On the University of Chicago campus.
Gold: the University of Chicago campus
-----
Q: What does the WG I Summary for Policymakers report say human activities are doing to greenhouse gases?
Pred: substantially increasing the atmospheric concentrations
Gold: substantially increasing the atmospheric concentrations
-----


In [9]:
import time

if "pred_en_basic" not in df.columns:
    df["pred_en_basic"] = ""

for i in range(len(df)):
    if isinstance(df.loc[i, "pred_en_basic"], str) and df.loc[i, "pred_en_basic"].strip():
        continue

    print(f"SQuAD EN-Basic {i+1}/{len(df)}")

    df.loc[i, "pred_en_basic"] = squad_en_basic(
        df.loc[i, "context"],
        df.loc[i, "question"]
    )

    if (i + 1) % 10 == 0:
        df.to_csv("squad_en_basic_progress.csv", index=False)

    time.sleep(0.3)

df.to_csv("squad_en_basic_final.csv", index=False)
print("Finished EN-Basic")

SQuAD EN-Basic 1/300
SQuAD EN-Basic 2/300
SQuAD EN-Basic 3/300
SQuAD EN-Basic 4/300
SQuAD EN-Basic 5/300
SQuAD EN-Basic 6/300
SQuAD EN-Basic 7/300
SQuAD EN-Basic 8/300
SQuAD EN-Basic 9/300
SQuAD EN-Basic 10/300
SQuAD EN-Basic 11/300
SQuAD EN-Basic 12/300
SQuAD EN-Basic 13/300
SQuAD EN-Basic 14/300
SQuAD EN-Basic 15/300
SQuAD EN-Basic 16/300
SQuAD EN-Basic 17/300
SQuAD EN-Basic 18/300
SQuAD EN-Basic 19/300
SQuAD EN-Basic 20/300
SQuAD EN-Basic 21/300
SQuAD EN-Basic 22/300
SQuAD EN-Basic 23/300
SQuAD EN-Basic 24/300
SQuAD EN-Basic 25/300
SQuAD EN-Basic 26/300
SQuAD EN-Basic 27/300
SQuAD EN-Basic 28/300
SQuAD EN-Basic 29/300
SQuAD EN-Basic 30/300
SQuAD EN-Basic 31/300
SQuAD EN-Basic 32/300
SQuAD EN-Basic 33/300
SQuAD EN-Basic 34/300
SQuAD EN-Basic 35/300
SQuAD EN-Basic 36/300
SQuAD EN-Basic 37/300
SQuAD EN-Basic 38/300
SQuAD EN-Basic 39/300
SQuAD EN-Basic 40/300
SQuAD EN-Basic 41/300
SQuAD EN-Basic 42/300
SQuAD EN-Basic 43/300
SQuAD EN-Basic 44/300
SQuAD EN-Basic 45/300
SQuAD EN-Basic 46/3

In [10]:
import re
import string
from collections import Counter
import pandas as pd

def normalize_answer(s):
    s = str(s).lower()
    s = re.sub(r'\b(a|an|the)\b', ' ', s)
    s = ''.join(ch for ch in s if ch not in string.punctuation)
    s = ' '.join(s.split())
    return s

def exact_match_score(prediction, ground_truth):
    return int(normalize_answer(prediction) == normalize_answer(ground_truth))

def f1_score_squad(prediction, ground_truth):
    pred_tokens = normalize_answer(prediction).split()
    gold_tokens = normalize_answer(ground_truth).split()

    common = Counter(pred_tokens) & Counter(gold_tokens)
    num_same = sum(common.values())

    if len(pred_tokens) == 0 or len(gold_tokens) == 0:
        return int(pred_tokens == gold_tokens)
    if num_same == 0:
        return 0

    precision = num_same / len(pred_tokens)
    recall = num_same / len(gold_tokens)
    return 2 * precision * recall / (precision + recall)

df_eval = pd.read_csv("squad_en_basic_final.csv")

em_scores = []
f1_scores = []

for _, row in df_eval.iterrows():
    em_scores.append(exact_match_score(row["pred_en_basic"], row["gold_answer"]))
    f1_scores.append(f1_score_squad(row["pred_en_basic"], row["gold_answer"]))

print("EM:", sum(em_scores) / len(em_scores))
print("F1:", sum(f1_scores) / len(f1_scores))

EM: 0.69
F1: 0.8546955568555518


In [11]:
import pandas as pd
import time
import os
from openai import OpenAI


input_file = "squad_val_300.csv"
progress_file = "squad_kz_progress.csv"
final_file = "squad_kz_300.csv"

if os.path.exists(progress_file):
    df = pd.read_csv(progress_file)
    print("Resuming from progress file...")
else:
    df = pd.read_csv(input_file)
    for col in ["context_kz", "question_kz", "gold_answer_kz"]:
        if col not in df.columns:
            df[col] = None

In [12]:
def translate_squad_row(context, question, answer):
    prompt = f"""Translate the following SQuAD sample into Kazakh.

Rules:
- Preserve the meaning exactly.
- Do not summarize.
- Translate the context, question, and answer into natural Kazakh.
- Keep the output in the exact format below.

FORMAT:
CONTEXT: <translated context>
QUESTION: <translated question>
ANSWER: <translated answer>

CONTEXT:
{context}

QUESTION:
{question}

ANSWER:
{answer}
"""

    response = client.chat.completions.create(
        model="gpt-4o",
        temperature=0,
        messages=[{"role": "user", "content": prompt}]
    )

    return response.choices[0].message.content.strip()

In [13]:
def split_squad_translation(text):
    lines = text.strip().split("\n")

    context_lines = []
    question = ""
    answer = ""

    mode = None
    for line in lines:
        if line.startswith("CONTEXT:"):
            mode = "context"
            context_lines.append(line.replace("CONTEXT:", "").strip())
        elif line.startswith("QUESTION:"):
            mode = "question"
            question = line.replace("QUESTION:", "").strip()
        elif line.startswith("ANSWER:"):
            mode = "answer"
            answer = line.replace("ANSWER:", "").strip()
        else:
            if mode == "context":
                context_lines.append(line.strip())

    context = "\n".join(context_lines).strip()
    return context, question, answer

In [14]:
for i in range(2):
    translated = translate_squad_row(
        df.loc[i, "context"],
        df.loc[i, "question"],
        df.loc[i, "gold_answer"]
    )
    print(translated)
    print("-----")

CONTEXT: Гипербарикалық (жоғары қысымды) медицина арнайы оттегі камераларын қолданып, науқастың және қажет болған жағдайда медициналық қызметкерлердің айналасындағы O2-нің ішінара қысымын арттырады. Көміртек тотығымен улану, газ гангренасы және декомпрессиялық ауру (бендс) кейде осы құрылғыларды қолдану арқылы емделеді. Өкпедегі O2 концентрациясының артуы гемоглобиннің гем тобына көміртек тотығын ығыстыруға көмектеседі. Оттегі газы газ гангренасын тудыратын анаэробты бактерияларға улы болып табылады, сондықтан оның ішінара қысымын арттыру оларды өлтіруге көмектеседі. Декомпрессиялық ауру сүңгуірлерде сүңгігеннен кейін тым тез декомпрессия жасағанда пайда болады, нәтижесінде олардың қанында инертті газдардың, негізінен азот пен гелийдің көпіршіктері пайда болады. O2 қысымын мүмкіндігінше тез арттыру емнің бір бөлігі болып табылады.

QUESTION: Науқастың өкпесіндегі оттегінің концентрациясының артуы нені ығыстырады?

ANSWER: көміртек тотығын
-----
КОНТЕКСТ:
Келісімдерге сәйкес, арналар үш

In [15]:
for i in range(len(df)):
    if isinstance(df.loc[i, "context_kz"], str) and df.loc[i, "context_kz"].strip():
        continue

    print(f"Translating {i+1}/{len(df)}")

    try:
        translated = translate_squad_row(
            df.loc[i, "context"],
            df.loc[i, "question"],
            df.loc[i, "gold_answer"]
        )

        context_kz, question_kz, answer_kz = split_squad_translation(translated)

        df.loc[i, "context_kz"] = context_kz
        df.loc[i, "question_kz"] = question_kz
        df.loc[i, "gold_answer_kz"] = answer_kz

    except Exception as e:
        print("Error:", e)
        df.to_csv(progress_file, index=False)
        print("Saved progress before stopping.")
        break

    if (i + 1) % 5 == 0:
        df.to_csv(progress_file, index=False)
        print("Progress saved")

    time.sleep(0.3)

df.to_csv(final_file, index=False)
print("Finished translation")

Translating 1/300
Translating 2/300
Translating 3/300
Translating 4/300
Translating 5/300
Progress saved
Translating 6/300
Translating 7/300
Translating 8/300
Translating 9/300
Translating 10/300
Progress saved
Translating 11/300
Translating 12/300
Translating 13/300
Translating 14/300
Translating 15/300
Progress saved
Translating 16/300
Translating 17/300
Translating 18/300
Translating 19/300
Translating 20/300
Progress saved
Translating 21/300
Translating 22/300
Translating 23/300
Translating 24/300
Translating 25/300
Progress saved
Translating 26/300
Translating 27/300
Translating 28/300
Translating 29/300
Translating 30/300
Progress saved
Translating 31/300
Translating 32/300
Translating 33/300
Translating 34/300
Translating 35/300
Progress saved
Translating 36/300
Translating 37/300
Translating 38/300
Translating 39/300
Translating 40/300
Progress saved
Translating 41/300
Translating 42/300
Translating 43/300
Translating 44/300
Translating 45/300
Progress saved
Translating 46/300


In [16]:
def squad_native_basic(context_kz, question_kz):
    prompt = f"""Мәтінді оқып, сұраққа мәтіннен алынған қысқа жауап бер.

Ережелер:
- Жауап мүмкіндігінше қысқа болсын.
- Толық сөйлем жазба.
- Түсіндірме қоспа.
- Егер жауап иә/жоқ болса, тек иә немесе жоқ деп жаз.

Мәтін:
{context_kz}

Сұрақ:
{question_kz}
"""

    response = client.chat.completions.create(
        model="gpt-4o",
        temperature=0,
        messages=[{"role": "user", "content": prompt}]
    )

    return response.choices[0].message.content.strip()

In [17]:
df = pd.read_csv("squad_kz_300.csv")

for i in range(3):
    pred = squad_native_basic(df.loc[i, "context_kz"], df.loc[i, "question_kz"])
    print("Q:", df.loc[i, "question_kz"])
    print("Pred:", pred)
    print("Gold:", df.loc[i, "gold_answer_kz"])
    print("-----")

Q: Науқастың өкпесіндегі оттегінің концентрациясының артуы нені ығыстырады?
Pred: Көміртек тотығын.
Gold: көміртек тотығы
-----
Q: nan
Pred: nan
Gold: nan
-----
Q: Шыңғыс ханның немересі Киев Русіне қай жылы басып кірді?
Pred: 1237 жылы.
Gold: 1237
-----


In [18]:
print(df.loc[1, ["context", "question", "gold_answer"]])
print(df.loc[1, ["context_kz", "question_kz", "gold_answer_kz"]])

context        The agreements include fixed annual carriage f...
question       Does the new deal include Video on demand and ...
gold_answer                                                   no
Name: 1, dtype: object
context_kz        NaN
question_kz       NaN
gold_answer_kz    NaN
Name: 1, dtype: object


In [22]:
print(df[["context_kz", "question_kz", "gold_answer_kz"]].isna().sum())

context_kz        122
question_kz       123
gold_answer_kz    123
dtype: int64


In [20]:
def split_squad_translation(text):
    context = ""
    question = ""
    answer = ""

    if "CONTEXT:" in text and "QUESTION:" in text and "ANSWER:" in text:
        context = text.split("CONTEXT:", 1)[1].split("QUESTION:", 1)[0].strip()
        question = text.split("QUESTION:", 1)[1].split("ANSWER:", 1)[0].strip()
        answer = text.split("ANSWER:", 1)[1].strip()

    return context, question, answer

In [21]:
import pandas as pd
import time

df = pd.read_csv("squad_kz_300.csv")

for i in range(len(df)):
    if pd.isna(df.loc[i, "context_kz"]) or pd.isna(df.loc[i, "question_kz"]) or pd.isna(df.loc[i, "gold_answer_kz"]):
        print(f"Fixing row {i+1}/{len(df)}")

        try:
            translated = translate_squad_row(
                df.loc[i, "context"],
                df.loc[i, "question"],
                df.loc[i, "gold_answer"]
            )

            context_kz, question_kz, answer_kz = split_squad_translation(translated)

            # only save if parser succeeded
            if context_kz and question_kz and answer_kz:
                df.loc[i, "context_kz"] = context_kz
                df.loc[i, "question_kz"] = question_kz
                df.loc[i, "gold_answer_kz"] = answer_kz
            else:
                print(f"Parsing failed at row {i}")

        except Exception as e:
            print(f"Error at row {i}: {e}")

        if (i + 1) % 5 == 0:
            df.to_csv("squad_kz_300.csv", index=False)
            print("Progress saved")

        time.sleep(0.3)

df.to_csv("squad_kz_300.csv", index=False)
print("Finished fixing missing rows.")

Fixing row 2/300
Parsing failed at row 1
Fixing row 6/300
Parsing failed at row 5
Fixing row 9/300
Parsing failed at row 8
Fixing row 10/300
Parsing failed at row 9
Progress saved
Fixing row 11/300
Fixing row 13/300
Parsing failed at row 12
Fixing row 17/300
Fixing row 19/300
Parsing failed at row 18
Fixing row 20/300
Parsing failed at row 19
Progress saved
Fixing row 21/300
Parsing failed at row 20
Fixing row 22/300
Parsing failed at row 21
Fixing row 24/300
Parsing failed at row 23
Fixing row 27/300
Parsing failed at row 26
Fixing row 29/300
Parsing failed at row 28
Fixing row 33/300
Parsing failed at row 32
Fixing row 34/300
Parsing failed at row 33
Fixing row 39/300
Parsing failed at row 38
Fixing row 42/300
Parsing failed at row 41
Fixing row 43/300
Parsing failed at row 42
Fixing row 44/300
Fixing row 46/300
Parsing failed at row 45
Fixing row 49/300
Fixing row 51/300
Parsing failed at row 50
Fixing row 53/300
Parsing failed at row 52
Fixing row 58/300
Parsing failed at row 57
Fi

In [23]:
def translate_squad_row(context, question, answer):
    prompt = f"""Translate the following into Kazakh.

IMPORTANT RULES:
- DO NOT translate the labels: CONTEXT, QUESTION, ANSWER
- Keep them EXACTLY in English
- Do NOT add any extra text
- Follow the format EXACTLY

FORMAT:
CONTEXT: <text>
QUESTION: <text>
ANSWER: <text>

CONTEXT:
{context}

QUESTION:
{question}

ANSWER:
{answer}
"""
    response = client.chat.completions.create(
        model="gpt-4o",
        temperature=0,
        messages=[{"role": "user", "content": prompt}]
    )

    return response.choices[0].message.content.strip()

In [24]:
def split_squad_translation(text):
    text = text.strip()

    try:
        context = text.split("CONTEXT:", 1)[1].split("QUESTION:", 1)[0].strip()
        question = text.split("QUESTION:", 1)[1].split("ANSWER:", 1)[0].strip()
        answer = text.split("ANSWER:", 1)[1].strip()

        return context, question, answer

    except:
        return "", "", ""

In [27]:
i=5
translated = translate_squad_row(
    df.loc[i, "context"],
    df.loc[i, "question"],
    df.loc[i, "gold_answer"]
)

print("RAW OUTPUT:")
print(translated)

RAW OUTPUT:
CONTEXT:
Амазон өзенінің ұзындығын алғаш рет еуропалық Франсиско де Орельяна 1542 жылы саяхаттаған. BBC-дің "Unnatural Histories" бағдарламасы Орельяна бұрын ойлағандай өз талаптарын асыра айтпағанын, керісінше, 1540 жылдары Амазон бойында күрделі өркениет гүлденіп жатқанын дұрыс байқағанын дәлелдейтін деректерді ұсынады. Бұл өркениет кейіннен Еуропадан келген шешек сияқты аурулардың таралуынан жойылған деп есептеледі. 1970 жылдардан бастап, AD 0–1250 жылдар аралығында ормансыздандырылған жерлерде көптеген геоглифтер табылып, Колумбқа дейінгі өркениеттер туралы талаптарды күшейтті. Геоглифтерді алғаш рет 1977 жылы Ондемар Диас ашқан және Алсеу Ранзи Акре үстінен ұшып өткеннен кейін олардың ашылуын жалғастырған. BBC-дің "Unnatural Histories" бағдарламасы Амазон орманының таза табиғи аймақ болмағанын, керісінше, орман бағбандығы және терра прета сияқты тәжірибелер арқылы кем дегенде 11,000 жыл бойы адам қолымен қалыптасқанын дәлелдейтін деректерді ұсынды.

QUESTION:
Амазонды 

In [28]:
print(df[["context_kz", "question_kz", "gold_answer_kz"]].isna().sum())

context_kz        122
question_kz       123
gold_answer_kz    123
dtype: int64


In [29]:
import re

def split_squad_translation_flexible(text: str):
    text = str(text).strip()

    # accept either English or Kazakh field labels
    patterns = [
        r"CONTEXT:\s*(.*?)\s*QUESTION:\s*(.*?)\s*ANSWER:\s*(.*)",
        r"Контекст:\s*(.*?)\s*Сұрақ:\s*(.*?)\s*Жауап:\s*(.*)",
        r"Мәтін:\s*(.*?)\s*Сұрақ:\s*(.*?)\s*Жауап:\s*(.*)",
    ]

    for p in patterns:
        m = re.search(p, text, flags=re.S | re.I)
        if m:
            context = m.group(1).strip()
            question = m.group(2).strip()
            answer = m.group(3).strip()
            return context, question, answer

    return "", "", ""

In [30]:
def translate_squad_row(context, question, answer):
    prompt = f"""Translate the following SQuAD sample into Kazakh.

STRICT RULES:
- Keep the labels EXACTLY as:
CONTEXT:
QUESTION:
ANSWER:
- Do not translate these labels.
- Do not add any extra text.
- Do not add notes or explanations.
- Output exactly 3 sections only.

CONTEXT:
{context}

QUESTION:
{question}

ANSWER:
{answer}
"""
    response = client.chat.completions.create(
        model="gpt-4o",
        temperature=0,
        messages=[{"role": "user", "content": prompt}]
    )
    return response.choices[0].message.content.strip()

In [31]:
import pandas as pd
import time

df = pd.read_csv("squad_kz_300.csv")

if "raw_translation" not in df.columns:
    df["raw_translation"] = None

for i in range(len(df)):
    missing = (
        pd.isna(df.loc[i, "context_kz"]) or
        pd.isna(df.loc[i, "question_kz"]) or
        pd.isna(df.loc[i, "gold_answer_kz"]) or
        str(df.loc[i, "context_kz"]).strip() == "" or
        str(df.loc[i, "question_kz"]).strip() == "" or
        str(df.loc[i, "gold_answer_kz"]).strip() == ""
    )

    if not missing:
        continue

    print(f"Fixing row {i+1}/{len(df)}")

    try:
        translated = translate_squad_row(
            df.loc[i, "context"],
            df.loc[i, "question"],
            df.loc[i, "gold_answer"]
        )

        df.loc[i, "raw_translation"] = translated

        context_kz, question_kz, answer_kz = split_squad_translation_flexible(translated)

        if context_kz and question_kz and answer_kz:
            df.loc[i, "context_kz"] = context_kz
            df.loc[i, "question_kz"] = question_kz
            df.loc[i, "gold_answer_kz"] = answer_kz
        else:
            print(f"Still failed at row {i}")

    except Exception as e:
        print(f"Error at row {i}: {e}")

    if (i + 1) % 5 == 0:
        df.to_csv("squad_kz_300.csv", index=False)
        print("Progress saved")

    time.sleep(0.3)

df.to_csv("squad_kz_300.csv", index=False)
print("Done")

Fixing row 2/300
Fixing row 6/300
Fixing row 9/300
Fixing row 10/300
Progress saved
Fixing row 13/300
Fixing row 19/300
Fixing row 20/300
Progress saved
Fixing row 21/300
Fixing row 22/300
Fixing row 24/300
Fixing row 27/300
Fixing row 29/300
Fixing row 33/300
Fixing row 34/300
Fixing row 39/300
Fixing row 42/300
Fixing row 43/300
Fixing row 46/300
Fixing row 51/300
Fixing row 53/300
Fixing row 58/300
Fixing row 64/300
Fixing row 66/300
Fixing row 69/300
Fixing row 71/300
Fixing row 72/300
Fixing row 73/300
Fixing row 74/300
Fixing row 76/300
Fixing row 77/300
Fixing row 78/300
Fixing row 79/300
Fixing row 82/300
Fixing row 83/300
Fixing row 85/300
Progress saved
Fixing row 86/300
Fixing row 89/300
Fixing row 95/300
Progress saved
Fixing row 99/300
Fixing row 101/300
Fixing row 104/300
Fixing row 107/300
Fixing row 108/300
Fixing row 109/300
Fixing row 110/300
Progress saved
Fixing row 111/300
Fixing row 113/300
Fixing row 114/300
Fixing row 115/300
Progress saved
Fixing row 116/300
Fi

In [32]:
print(df[["context_kz", "question_kz", "gold_answer_kz"]].isna().sum())

context_kz        0
question_kz       0
gold_answer_kz    0
dtype: int64


In [33]:
def squad_native_basic(context_kz, question_kz):
    prompt = f"""Мәтінді оқып, сұраққа мәтіннен алынған қысқа жауап бер.

Ережелер:
- Жауап мүмкіндігінше қысқа болсын.
- Толық сөйлем жазба.
- Түсіндірме қоспа.
- Егер жауап иә немесе жоқ болса, тек иә немесе жоқ деп жаз.

Мәтін:
{context_kz}

Сұрақ:
{question_kz}
"""
    response = client.chat.completions.create(
        model="gpt-4o",
        temperature=0,
        messages=[{"role": "user", "content": prompt}]
    )

    return response.choices[0].message.content.strip()

In [34]:
import pandas as pd

df = pd.read_csv("squad_kz_300.csv")

for i in range(3):
    pred = squad_native_basic(df.loc[i, "context_kz"], df.loc[i, "question_kz"])
    print("Q:", df.loc[i, "question_kz"])
    print("Pred:", pred)
    print("Gold:", df.loc[i, "gold_answer_kz"])
    print("-----")

Q: Науқастың өкпесіндегі оттегінің концентрациясының артуы нені ығыстырады?
Pred: Көміртек тотығын.
Gold: көміртек тотығы
-----
Q: Жаңа келісім сұраныс бойынша бейне және жоғары анықтамалықты қамтиды ма?
Pred: Ешқандай белгі жоқ.
Gold: жоқ
-----
Q: Шыңғыс ханның немересі Киев Русіне қай жылы басып кірді?
Pred: 1237 жылы.
Gold: 1237
-----


In [35]:
import time

if "pred_native_basic" not in df.columns:
    df["pred_native_basic"] = ""

for i in range(len(df)):
    if isinstance(df.loc[i, "pred_native_basic"], str) and df.loc[i, "pred_native_basic"].strip():
        continue

    print(f"SQuAD Native-Basic {i+1}/{len(df)}")

    df.loc[i, "pred_native_basic"] = squad_native_basic(
        df.loc[i, "context_kz"],
        df.loc[i, "question_kz"]
    )

    if (i + 1) % 10 == 0:
        df.to_csv("squad_native_basic_progress.csv", index=False)

    time.sleep(0.3)

df.to_csv("squad_native_basic_final.csv", index=False)
print("Finished Native-Basic")

SQuAD Native-Basic 1/300
SQuAD Native-Basic 2/300
SQuAD Native-Basic 3/300
SQuAD Native-Basic 4/300
SQuAD Native-Basic 5/300
SQuAD Native-Basic 6/300
SQuAD Native-Basic 7/300
SQuAD Native-Basic 8/300
SQuAD Native-Basic 9/300
SQuAD Native-Basic 10/300
SQuAD Native-Basic 11/300
SQuAD Native-Basic 12/300
SQuAD Native-Basic 13/300
SQuAD Native-Basic 14/300
SQuAD Native-Basic 15/300
SQuAD Native-Basic 16/300
SQuAD Native-Basic 17/300
SQuAD Native-Basic 18/300
SQuAD Native-Basic 19/300
SQuAD Native-Basic 20/300
SQuAD Native-Basic 21/300
SQuAD Native-Basic 22/300
SQuAD Native-Basic 23/300
SQuAD Native-Basic 24/300
SQuAD Native-Basic 25/300
SQuAD Native-Basic 26/300
SQuAD Native-Basic 27/300
SQuAD Native-Basic 28/300
SQuAD Native-Basic 29/300
SQuAD Native-Basic 30/300
SQuAD Native-Basic 31/300
SQuAD Native-Basic 32/300
SQuAD Native-Basic 33/300
SQuAD Native-Basic 34/300
SQuAD Native-Basic 35/300
SQuAD Native-Basic 36/300
SQuAD Native-Basic 37/300
SQuAD Native-Basic 38/300
SQuAD Native-Basic 39

In [36]:
import re
import string
from collections import Counter
import pandas as pd

def normalize_answer(s):
    s = str(s).lower()
    s = ''.join(ch for ch in s if ch not in string.punctuation)
    s = ' '.join(s.split())
    return s

def exact_match_score(prediction, ground_truth):
    return int(normalize_answer(prediction) == normalize_answer(ground_truth))

def f1_score_squad(prediction, ground_truth):
    pred_tokens = normalize_answer(prediction).split()
    gold_tokens = normalize_answer(ground_truth).split()

    common = Counter(pred_tokens) & Counter(gold_tokens)
    num_same = sum(common.values())

    if len(pred_tokens) == 0 or len(gold_tokens) == 0:
        return int(pred_tokens == gold_tokens)
    if num_same == 0:
        return 0

    precision = num_same / len(pred_tokens)
    recall = num_same / len(gold_tokens)
    return 2 * precision * recall / (precision + recall)

In [37]:
df_eval = pd.read_csv("squad_native_basic_final.csv")

em_scores = []
f1_scores = []

for _, row in df_eval.iterrows():
    em_scores.append(exact_match_score(row["pred_native_basic"], row["gold_answer_kz"]))
    f1_scores.append(f1_score_squad(row["pred_native_basic"], row["gold_answer_kz"]))

print("EM:", sum(em_scores) / len(em_scores))
print("F1:", sum(f1_scores) / len(f1_scores))

EM: 0.5066666666666667
F1: 0.7336138545550309


In [38]:
def squad_cross_lingual(context_kz, question_kz):
    prompt = f"""Read the context and answer the question using only a short answer span.

Rules:
- Return only the shortest possible answer phrase.
- Do not write a full sentence.
- Do not explain.
- If the answer is yes or no, return only yes or no.

Context:
{context_kz}

Question:
{question_kz}
"""
    response = client.chat.completions.create(
        model="gpt-4o",
        temperature=0,
        messages=[{"role": "user", "content": prompt}]
    )

    return response.choices[0].message.content.strip()

In [39]:
df = pd.read_csv("squad_kz_300.csv")

for i in range(3):
    pred = squad_cross_lingual(df.loc[i, "context_kz"], df.loc[i, "question_kz"])
    print("Q:", df.loc[i, "question_kz"])
    print("Pred:", pred)
    print("Gold:", df.loc[i, "gold_answer_kz"])
    print("-----")

Q: Науқастың өкпесіндегі оттегінің концентрациясының артуы нені ығыстырады?
Pred: көміртек тотығын
Gold: көміртек тотығы
-----
Q: Жаңа келісім сұраныс бойынша бейне және жоғары анықтамалықты қамтиды ма?
Pred: ешқандай белгі жоқ
Gold: жоқ
-----
Q: Шыңғыс ханның немересі Киев Русіне қай жылы басып кірді?
Pred: 1237 жылы
Gold: 1237
-----


In [40]:
if "pred_cross_lingual" not in df.columns:
    df["pred_cross_lingual"] = ""

for i in range(len(df)):
    if isinstance(df.loc[i, "pred_cross_lingual"], str) and df.loc[i, "pred_cross_lingual"].strip():
        continue

    print(f"SQuAD Cross-lingual {i+1}/{len(df)}")

    df.loc[i, "pred_cross_lingual"] = squad_cross_lingual(
        df.loc[i, "context_kz"],
        df.loc[i, "question_kz"]
    )

    if (i + 1) % 10 == 0:
        df.to_csv("squad_cross_lingual_progress.csv", index=False)

    time.sleep(0.3)

df.to_csv("squad_cross_lingual_final.csv", index=False)
print("Finished Cross-lingual")

SQuAD Cross-lingual 1/300
SQuAD Cross-lingual 2/300
SQuAD Cross-lingual 3/300
SQuAD Cross-lingual 4/300
SQuAD Cross-lingual 5/300
SQuAD Cross-lingual 6/300
SQuAD Cross-lingual 7/300
SQuAD Cross-lingual 8/300
SQuAD Cross-lingual 9/300
SQuAD Cross-lingual 10/300
SQuAD Cross-lingual 11/300
SQuAD Cross-lingual 12/300
SQuAD Cross-lingual 13/300
SQuAD Cross-lingual 14/300
SQuAD Cross-lingual 15/300
SQuAD Cross-lingual 16/300
SQuAD Cross-lingual 17/300
SQuAD Cross-lingual 18/300
SQuAD Cross-lingual 19/300
SQuAD Cross-lingual 20/300
SQuAD Cross-lingual 21/300
SQuAD Cross-lingual 22/300
SQuAD Cross-lingual 23/300
SQuAD Cross-lingual 24/300
SQuAD Cross-lingual 25/300
SQuAD Cross-lingual 26/300
SQuAD Cross-lingual 27/300
SQuAD Cross-lingual 28/300
SQuAD Cross-lingual 29/300
SQuAD Cross-lingual 30/300
SQuAD Cross-lingual 31/300
SQuAD Cross-lingual 32/300
SQuAD Cross-lingual 33/300
SQuAD Cross-lingual 34/300
SQuAD Cross-lingual 35/300
SQuAD Cross-lingual 36/300
SQuAD Cross-lingual 37/300
SQuAD Cros

SQuAD Cross-lingual 298/300
SQuAD Cross-lingual 299/300
SQuAD Cross-lingual 300/300
Finished Cross-lingual


In [42]:
df_eval = pd.read_csv("squad_cross_lingual_final.csv")

em_scores = []
f1_scores = []

for _, row in df_eval.iterrows():
    em_scores.append(exact_match_score(row["pred_cross_lingual"], row["gold_answer_kz"]))
    f1_scores.append(f1_score_squad(row["pred_cross_lingual"], row["gold_answer_kz"]))

print("EM:", sum(em_scores) / len(em_scores))
print("F1:", sum(f1_scores) / len(f1_scores))

EM: 0.53
F1: 0.738514025246784


In [44]:
def squad_translate_en(context_kz, question_kz):
    prompt = f"""Translate the context and question into English internally, answer the question, and return only the final answer in Kazakh.

STRICT RULES:
1. Do NOT output the English translation.
2. Do NOT explain.
3. Output ONLY the answer text.
4. The answer must be as short as possible.
5. The answer must be in Kazakh.
6. If yes/no, answer only: иә or жоқ.

Context:
{context_kz}

Question:
{question_kz}
"""
    response = client.chat.completions.create(
        model="gpt-4o",
        temperature=0,
        messages=[{"role": "user", "content": prompt}]
    )

    return response.choices[0].message.content.strip()

In [45]:
df = pd.read_csv("squad_kz_300.csv")

for i in range(3):
    pred = squad_translate_en(df.loc[i, "context_kz"], df.loc[i, "question_kz"])
    print("Q:", df.loc[i, "question_kz"])
    print("Pred:", pred)
    print("Gold:", df.loc[i, "gold_answer_kz"])
    print("-----")

Q: Науқастың өкпесіндегі оттегінің концентрациясының артуы нені ығыстырады?
Pred: көміртек тотығын
Gold: көміртек тотығы
-----
Q: Жаңа келісім сұраныс бойынша бейне және жоғары анықтамалықты қамтиды ма?
Pred: Жоқ.
Gold: жоқ
-----
Q: Шыңғыс ханның немересі Киев Русіне қай жылы басып кірді?
Pred: 1237 жылы.
Gold: 1237
-----


In [46]:
if "pred_translate_en" not in df.columns:
    df["pred_translate_en"] = ""

for i in range(len(df)):
    if isinstance(df.loc[i, "pred_translate_en"], str) and df.loc[i, "pred_translate_en"].strip():
        continue

    print(f"SQuAD Translate-En {i+1}/{len(df)}")

    df.loc[i, "pred_translate_en"] = squad_translate_en(
        df.loc[i, "context_kz"],
        df.loc[i, "question_kz"]
    )

    if (i + 1) % 10 == 0:
        df.to_csv("squad_translate_en_progress.csv", index=False)

    time.sleep(0.3)

df.to_csv("squad_translate_en_final.csv", index=False)
print("Finished Translate-En")

SQuAD Translate-En 1/300
SQuAD Translate-En 2/300
SQuAD Translate-En 3/300
SQuAD Translate-En 4/300
SQuAD Translate-En 5/300
SQuAD Translate-En 6/300
SQuAD Translate-En 7/300
SQuAD Translate-En 8/300
SQuAD Translate-En 9/300
SQuAD Translate-En 10/300
SQuAD Translate-En 11/300
SQuAD Translate-En 12/300
SQuAD Translate-En 13/300
SQuAD Translate-En 14/300
SQuAD Translate-En 15/300
SQuAD Translate-En 16/300
SQuAD Translate-En 17/300
SQuAD Translate-En 18/300
SQuAD Translate-En 19/300
SQuAD Translate-En 20/300
SQuAD Translate-En 21/300
SQuAD Translate-En 22/300
SQuAD Translate-En 23/300
SQuAD Translate-En 24/300
SQuAD Translate-En 25/300
SQuAD Translate-En 26/300
SQuAD Translate-En 27/300
SQuAD Translate-En 28/300
SQuAD Translate-En 29/300
SQuAD Translate-En 30/300
SQuAD Translate-En 31/300
SQuAD Translate-En 32/300
SQuAD Translate-En 33/300
SQuAD Translate-En 34/300
SQuAD Translate-En 35/300
SQuAD Translate-En 36/300
SQuAD Translate-En 37/300
SQuAD Translate-En 38/300
SQuAD Translate-En 39

In [47]:
df_eval = pd.read_csv("squad_translate_en_final.csv")

em_scores = []
f1_scores = []

for _, row in df_eval.iterrows():
    em_scores.append(exact_match_score(row["pred_translate_en"], row["gold_answer_kz"]))
    f1_scores.append(f1_score_squad(row["pred_translate_en"], row["gold_answer_kz"]))

print("EM:", sum(em_scores) / len(em_scores))
print("F1:", sum(f1_scores) / len(f1_scores))

EM: 0.51
F1: 0.7145757822424488
